# 02_v2 Preprocessing Policy Validation

This notebook validates preprocessing policy candidates for the active v2 OTT churn dataset. It does not create interim datasets, usage features, modeling tables, or trained models.


In [1]:
import csv
import json
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "_data" / "01_raw"
STAGE01_TABLE_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "tables" / "01_v2_data_overview_and_audit"
STAGE01_DATA_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "01_v2_data_overview_and_audit"
TABLE_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "tables" / "02_v2_preprocessing_policy_validation"
DATA_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "02_v2_preprocessing_policy_validation"

RAW_FILES = {
    "Membership": RAW_DIR / "Membership.csv",
    "User_Mapping": RAW_DIR / "User_Mapping.csv",
    "View_History": RAW_DIR / "View_History.csv",
    "Movie_Master": RAW_DIR / "Movie_Master.csv",
}
STAGE01_REQUIRED = [
    STAGE01_TABLE_DIR / "01_v2_raw_file_inventory.csv",
    STAGE01_TABLE_DIR / "01_v2_schema_summary.csv",
    STAGE01_TABLE_DIR / "01_v2_membership_duplicate_audit.csv",
    STAGE01_TABLE_DIR / "01_v2_membership_target_conflict_rows.csv",
    STAGE01_TABLE_DIR / "01_v2_membership_duration_distribution.csv",
    STAGE01_TABLE_DIR / "01_v2_membership_value_anomaly_summary.csv",
    STAGE01_TABLE_DIR / "01_v2_usermapping_cardinality_audit.csv",
    STAGE01_TABLE_DIR / "01_v2_membership_view_temporal_audit.csv",
    STAGE01_TABLE_DIR / "01_v2_join_expansion_summary.csv",
    STAGE01_DATA_DIR / "01_v2_data_audit_report.md",
    STAGE01_DATA_DIR / "01_v2_audit_summary.json",
]
TARGET_COL = "is_repurchase"
CORE_EVENT_FIELDS = ["USER_KEY", "product_code", "price", "max_screen", "reg_date", "end_date", "payment_device", "billing_method"]
EXPECTED_SCREEN_VALUES = {1, 2, 3, 4}

TABLE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
raw_before = {name: {"size": path.stat().st_size, "mtime_ns": path.stat().st_mtime_ns} for name, path in RAW_FILES.items()}

def rel(path):
    return str(path.relative_to(PROJECT_ROOT)).replace("\\", "/")

def read_csv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        return reader.fieldnames or [], list(reader)

def write_csv(path, rows, fieldnames):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({field: row.get(field, "") for field in fieldnames})

def write_json(path, payload):
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

def parse_date(value, fmt):
    return datetime.strptime(value, fmt).date()

def safe_int(value):
    try:
        return int(value)
    except Exception:
        return None

def safe_float(value):
    try:
        return float(value)
    except Exception:
        return None

def sample_rows(rows, limit=20):
    return rows[:limit]

def add_reason(reason_rows, membership_row_id, reason_code, source_rule_id, severity, detail):
    reason_rows.append({
        "membership_row_id": membership_row_id,
        "reason_code": reason_code,
        "source_rule_id": source_rule_id,
        "severity": severity,
        "detail": detail,
    })

# Re-read Stage 01 outputs and active raw files.
stage01_inventory = []
for path in STAGE01_REQUIRED:
    stage01_inventory.append({
        "relative_path": rel(path),
        "exists": path.exists(),
        "file_size_bytes": path.stat().st_size if path.exists() else "",
        "last_write_time": datetime.fromtimestamp(path.stat().st_mtime).isoformat() if path.exists() else "",
        "role": "stage01_input_reference",
    })

loaded = {}
for name, path in RAW_FILES.items():
    cols, rows = read_csv(path)
    loaded[name] = {"columns": cols, "rows": rows, "path": path}
    stage01_inventory.append({
        "relative_path": rel(path),
        "exists": path.exists(),
        "file_size_bytes": path.stat().st_size,
        "last_write_time": datetime.fromtimestamp(path.stat().st_mtime).isoformat(),
        "role": "active_v2_raw_truth",
    })

membership_cols = loaded["Membership"]["columns"]
membership = []
for idx, row in enumerate(loaded["Membership"]["rows"], start=1):
    item = dict(row)
    item["membership_row_id"] = idx
    try:
        reg = parse_date(item["reg_date"], "%y-%m-%d")
        end = parse_date(item["end_date"], "%y-%m-%d")
        item["duration_days_calc"] = (end - reg).days
        item["date_parse_ok"] = True
    except Exception:
        item["duration_days_calc"] = ""
        item["date_parse_ok"] = False
    membership.append(item)
mapping = loaded["User_Mapping"]["rows"]
views = loaded["View_History"]["rows"]
movies = loaded["Movie_Master"]["rows"]

# Duplicate and strict conflict validation with sample rows.
full_groups = defaultdict(list)
non_target_cols = [c for c in membership_cols if c != TARGET_COL]
non_target_groups = defaultdict(list)
user_key_groups = defaultdict(list)
core_event_groups = defaultdict(list)
for row in membership:
    full_groups[tuple(row.get(c, "") for c in membership_cols)].append(row)
    non_target_groups[tuple(row.get(c, "") for c in non_target_cols)].append(row)
    user_key_groups[row["USER_KEY"]].append(row)
    core_event_groups[tuple(row.get(c, "") for c in CORE_EVENT_FIELDS)].append(row)

strict_conflict_groups = {key: rows for key, rows in non_target_groups.items() if len({r[TARGET_COL] for r in rows}) > 1}
exact_duplicate_groups = {key: rows for key, rows in full_groups.items() if len(rows) > 1}
non_target_duplicate_groups = {key: rows for key, rows in non_target_groups.items() if len(rows) > 1}
core_event_ambiguous_groups = {}
secondary_cols = [c for c in membership_cols if c not in CORE_EVENT_FIELDS + [TARGET_COL]]
for key, rows in core_event_groups.items():
    if len(rows) <= 1:
        continue
    target_values = {r[TARGET_COL] for r in rows}
    secondary_values = {tuple(r.get(c, "") for c in secondary_cols) for r in rows}
    if len(target_values) > 1 or len(secondary_values) > 1:
        core_event_ambiguous_groups[key] = rows

strict_conflict_group_audit = []
strict_conflict_sample_rows = []
for group_id, (key, rows) in enumerate(strict_conflict_groups.items(), start=1):
    target_counts = Counter(r[TARGET_COL] for r in rows)
    strict_conflict_group_audit.append({
        "conflict_group_id": group_id,
        "group_size": len(rows),
        "target_values": "|".join(sorted(target_counts)),
        "target_counts": "|".join(f"{k}:{v}" for k, v in sorted(target_counts.items())),
        "membership_row_ids": "|".join(str(r["membership_row_id"]) for r in rows),
        "recommended_action": "ask_mentor_or_exclude_from_training_until_business_rule_exists",
        "reason_code": "STRICT_TARGET_CONFLICT",
    })
    for r in rows[:5]:
        out = {"conflict_group_id": group_id, "sample_type": "strict_target_conflict"}
        out.update({"membership_row_id": r["membership_row_id"]})
        for c in membership_cols:
            out[c] = r.get(c, "")
        strict_conflict_sample_rows.append(out)

duplicate_group_audit = [
    {"duplicate_type": "exact_duplicate_all_columns", "group_count": len(exact_duplicate_groups), "row_count": sum(len(v) for v in exact_duplicate_groups.values()), "extra_rows": sum(len(v)-1 for v in exact_duplicate_groups.values()), "recommendation": "candidate_flag_or_deduplicate_after_target_policy_review", "reason_code": "EXACT_DUPLICATE_ROW"},
    {"duplicate_type": "duplicate_excluding_is_repurchase", "group_count": len(non_target_duplicate_groups), "row_count": sum(len(v) for v in non_target_duplicate_groups.values()), "extra_rows": sum(len(v)-1 for v in non_target_duplicate_groups.values()), "recommendation": "split_strict_conflict_from_same_target_duplicates", "reason_code": "DUPLICATE_EXCLUDING_TARGET"},
    {"duplicate_type": "same_USER_KEY_multiple_membership_events", "group_count": sum(1 for v in user_key_groups.values() if len(v) > 1), "row_count": sum(len(v) for v in user_key_groups.values() if len(v) > 1), "extra_rows": sum(len(v)-1 for v in user_key_groups.values() if len(v) > 1), "recommendation": "keep_as_multiple_subscription_events_unless_event_fields_prove_duplicate", "reason_code": "MULTIPLE_MEMBERSHIP_EVENTS_SAME_USER_KEY"},
    {"duplicate_type": "core_event_ambiguous", "group_count": len(core_event_ambiguous_groups), "row_count": sum(len(v) for v in core_event_ambiguous_groups.values()), "extra_rows": sum(len(v)-1 for v in core_event_ambiguous_groups.values()), "recommendation": "flag_for_manual_policy_review", "reason_code": "CORE_EVENT_AMBIGUOUS"},
]
duplicate_group_sample_rows = []
for duplicate_type, groups in [("exact_duplicate_all_columns", exact_duplicate_groups), ("duplicate_excluding_is_repurchase", non_target_duplicate_groups), ("core_event_ambiguous", core_event_ambiguous_groups)]:
    for group_id, rows in enumerate(list(groups.values())[:20], start=1):
        for r in rows[:4]:
            out = {"duplicate_type": duplicate_type, "group_id": group_id, "membership_row_id": r["membership_row_id"]}
            for c in membership_cols:
                out[c] = r.get(c, "")
            duplicate_group_sample_rows.append(out)

# Candidate policies and row-level reason candidates.
candidate_reason_rows = []
strict_conflict_ids = {r["membership_row_id"] for rows in strict_conflict_groups.values() for r in rows}
exact_duplicate_extra_ids = set()
for rows in exact_duplicate_groups.values():
    for r in sorted(rows, key=lambda x: x["membership_row_id"])[1:]:
        exact_duplicate_extra_ids.add(r["membership_row_id"])
core_ambiguous_ids = {r["membership_row_id"] for rows in core_event_ambiguous_groups.values() for r in rows}

for mid in sorted(strict_conflict_ids):
    add_reason(candidate_reason_rows, mid, "STRICT_TARGET_CONFLICT", "R_LABEL_01", "exclude_candidate", "All non-target Membership fields are identical but is_repurchase differs.")
for mid in sorted(exact_duplicate_extra_ids):
    add_reason(candidate_reason_rows, mid, "EXACT_DUPLICATE_EXTRA_ROW", "R_DUP_01", "deduplicate_candidate", "Exact duplicate extra row after retaining first observed row within duplicate group.")
for mid in sorted(core_ambiguous_ids - strict_conflict_ids):
    add_reason(candidate_reason_rows, mid, "CORE_EVENT_AMBIGUOUS", "R_DUP_02", "flag_candidate", "Core subscription fields match but target or secondary attributes differ.")

# Duration policies, tested without committing.
duration_policies = {
    "DUR_KEEP_ALL_PARSEABLE": lambda d: d != "" and isinstance(d, int),
    "DUR_POSITIVE_ONLY": lambda d: isinstance(d, int) and d > 0,
    "DUR_31_OR_32_ONLY": lambda d: d in {31, 32},
    "DUR_28_TO_35_ONLY": lambda d: isinstance(d, int) and 28 <= d <= 35,
}
duration_policy_comparison = []
duration_policy_samples = []
duration_counter = Counter(r["duration_days_calc"] for r in membership)
for policy_id, predicate in duration_policies.items():
    kept = [r for r in membership if predicate(r["duration_days_calc"])]
    excluded = [r for r in membership if not predicate(r["duration_days_calc"])]
    duration_policy_comparison.append({
        "policy_id": policy_id,
        "policy_type": "candidate_not_applied",
        "before_rows": len(membership),
        "kept_rows": len(kept),
        "excluded_rows": len(excluded),
        "excluded_rate": round(len(excluded)/len(membership), 6),
        "duration_distribution": "|".join(f"{k}:{v}" for k, v in sorted(duration_counter.items(), key=lambda x: str(x[0]))),
        "recommendation": "defer_final_duration_policy_until_end_date_inclusiveness_and_subscription_definition_are_confirmed",
    })
    for r in excluded[:30]:
        duration_policy_samples.append({
            "policy_id": policy_id,
            "membership_row_id": r["membership_row_id"],
            "reason_code": "DURATION_POLICY_EXCLUSION_CANDIDATE",
            "duration_days": r["duration_days_calc"],
            "USER_KEY": r["USER_KEY"],
            "reg_date": r["reg_date"],
            "end_date": r["end_date"],
            "is_repurchase": r[TARGET_COL],
        })

# UserMapping one-to-many and many-to-one audit.
key_to_nums = defaultdict(list)
num_to_keys = defaultdict(list)
for row in mapping:
    key_to_nums[row["USER_KEY"]].append(row["USER_NUM"])
    num_to_keys[row["USER_NUM"]].append(row["USER_KEY"])
membership_by_user_key_count = Counter(r["USER_KEY"] for r in membership)
usermapping_one_to_many_audit = []
for user_key, nums in key_to_nums.items():
    distinct_nums = sorted(set(nums), key=lambda x: safe_int(x) if safe_int(x) is not None else x)
    if len(distinct_nums) > 1:
        usermapping_one_to_many_audit.append({
            "USER_KEY": user_key,
            "mapping_row_count": len(nums),
            "distinct_USER_NUM_count": len(distinct_nums),
            "USER_NUM_values": "|".join(distinct_nums),
            "membership_event_count_for_USER_KEY": membership_by_user_key_count[user_key],
            "would_duplicate_membership_rows_on_direct_join": membership_by_user_key_count[user_key] * (len(distinct_nums) - 1),
            "recommendation": "do_not_directly_expand_final_modeling_rows; aggregate mapped USER_NUM logs back to membership_row_id",
            "decision_candidate": "flag",
        })
usermapping_many_to_one_audit = []
for user_num, keys in num_to_keys.items():
    distinct_keys = sorted(set(keys))
    if len(distinct_keys) > 1:
        usermapping_many_to_one_audit.append({
            "USER_NUM": user_num,
            "mapping_row_count": len(keys),
            "distinct_USER_KEY_count": len(distinct_keys),
            "USER_KEY_values": "|".join(distinct_keys),
            "recommendation": "ask_mentor_before_using_USER_NUM_as_group_key",
            "decision_candidate": "ask_mentor",
        })

# Value anomaly audit.
value_anomaly_audit = []
value_anomaly_samples = []
def add_anomaly(rule_id, field, reason_code, predicate, recommendation, decision_candidate):
    affected = [r for r in membership if predicate(r)]
    value_anomaly_audit.append({
        "rule_id": rule_id,
        "field": field,
        "reason_code": reason_code,
        "affected_rows": len(affected),
        "affected_rate": round(len(affected)/len(membership), 6),
        "decision_candidate": decision_candidate,
        "recommendation": recommendation,
    })
    for r in affected[:25]:
        value_anomaly_samples.append({
            "rule_id": rule_id,
            "field": field,
            "reason_code": reason_code,
            "membership_row_id": r["membership_row_id"],
            "USER_KEY": r["USER_KEY"],
            "value": r.get(field, ""),
            "is_repurchase": r[TARGET_COL],
            "recommendation": recommendation,
        })
    return affected
add_anomaly("R_AGE_01", "age", "AGE_MISSING_OR_NON_NUMERIC", lambda r: safe_int(r.get("age", "")) is None, "flag; do not impute silently", "flag")
add_anomaly("R_AGE_02", "age", "AGE_OUTSIDE_0_100", lambda r: (safe_int(r.get("age", "")) is not None and (safe_int(r.get("age", "")) < 0 or safe_int(r.get("age", "")) > 100)), "ask mentor if age range is encoded or true age", "ask_mentor")
add_anomaly("R_SCREEN_01", "max_screen", "MAX_SCREEN_MISSING_OR_NON_NUMERIC", lambda r: safe_int(r.get("max_screen", "")) is None, "flag; do not impute silently", "flag")
add_anomaly("R_SCREEN_02", "max_screen", "MAX_SCREEN_OUTSIDE_EXPECTED_1_4", lambda r: safe_int(r.get("max_screen", "")) is not None and safe_int(r.get("max_screen", "")) not in EXPECTED_SCREEN_VALUES, "ask mentor whether values outside 1-4 are valid products", "ask_mentor")
add_anomaly("R_GENDER_01", "gender", "GENDER_MISSING", lambda r: r.get("gender", "") == "", "flag missing demographic; do not exclude by default", "flag")
add_anomaly("R_GENDER_02", "gender", "GENDER_UNEXPECTED_CATEGORY", lambda r: r.get("gender", "") not in {"", "M", "F"}, "ask mentor for codebook", "ask_mentor")
add_anomaly("R_VERIFY_01", "is_user_verified", "VERIFICATION_MISSING", lambda r: r.get("is_user_verified", "") == "", "flag missing verification; do not exclude by default", "flag")
add_anomaly("R_VERIFY_02", "is_user_verified", "VERIFICATION_UNEXPECTED_CATEGORY", lambda r: r.get("is_user_verified", "") not in {"", "Y", "N"}, "ask mentor for codebook", "ask_mentor")
add_anomaly("R_PRICE_01", "price", "PRICE_MISSING_OR_NON_NUMERIC", lambda r: safe_float(r.get("price", "")) is None, "ask mentor; price is likely core subscription field", "ask_mentor")
add_anomaly("R_PRICE_02", "price", "PRICE_NEGATIVE", lambda r: safe_float(r.get("price", "")) is not None and safe_float(r.get("price", "")) < 0, "ask mentor before excluding", "ask_mentor")
add_anomaly("R_PROMO_01", "is_promotion", "PROMOTION_MISSING", lambda r: r.get("is_promotion", "") == "", "flag as non-explicit promotion status until codebook confirms blank meaning", "flag")
add_anomaly("R_PROMO_02", "is_promotion", "PROMOTION_UNEXPECTED_CATEGORY", lambda r: r.get("is_promotion", "") not in {"", "O"}, "ask mentor for codebook", "ask_mentor")
add_anomaly("R_PROMO_03", "is_promotion", "PRICE_100_PROMOTION_MISMATCH", lambda r: r.get("price", "") == "100" and r.get("is_promotion", "") != "O", "ask mentor whether price=100 defines promotion", "ask_mentor")
add_anomaly("R_CHURNPREVENT_01", "is_churn_prevented", "CHURN_PREVENTED_MISSING", lambda r: r.get("is_churn_prevented", "") == "", "flag; do not use to correct target without business rule", "flag")
add_anomaly("R_CHURNPREVENT_02", "is_churn_prevented", "CHURN_PREVENTED_UNEXPECTED_CATEGORY", lambda r: r.get("is_churn_prevented", "") not in {"", "O"}, "ask mentor for codebook", "ask_mentor")

# Join expansion under candidate membership cleaning policies.
views_by_user_num_count = Counter(v["USER_NUM"] for v in views)
user_nums_by_user_key = defaultdict(set)
for row in mapping:
    user_nums_by_user_key[row["USER_KEY"]].add(row["USER_NUM"])
def joined_count_for_membership_ids(kept_ids):
    rows = [r for r in membership if r["membership_row_id"] in kept_ids]
    total = 0
    multi_usernum_membership_rows = 0
    no_usernum_rows = 0
    for r in rows:
        nums = user_nums_by_user_key.get(r["USER_KEY"], set())
        if not nums:
            no_usernum_rows += 1
        if len(nums) > 1:
            multi_usernum_membership_rows += 1
        total += sum(views_by_user_num_count[n] for n in nums)
    return total, multi_usernum_membership_rows, no_usernum_rows
all_ids = {r["membership_row_id"] for r in membership}
policy_id_sets = {
    "M_KEEP_ALL": all_ids,
    "M_EXCLUDE_STRICT_TARGET_CONFLICT": all_ids - strict_conflict_ids,
    "M_DROP_EXACT_DUPLICATE_EXTRAS_ONLY": all_ids - exact_duplicate_extra_ids,
    "M_EXCLUDE_STRICT_AND_DROP_EXACT_EXTRAS": all_ids - strict_conflict_ids - exact_duplicate_extra_ids,
    "M_KEEP_DURATION_31_OR_32_ONLY": {r["membership_row_id"] for r in membership if r["duration_days_calc"] in {31, 32}},
    "M_KEEP_DURATION_28_TO_35_ONLY": {r["membership_row_id"] for r in membership if isinstance(r["duration_days_calc"], int) and 28 <= r["duration_days_calc"] <= 35},
}
join_expansion_by_policy = []
for policy_id, kept_ids in policy_id_sets.items():
    joined_count, multi_usernum_rows, no_usernum_rows = joined_count_for_membership_ids(kept_ids)
    join_expansion_by_policy.append({
        "policy_id": policy_id,
        "policy_type": "candidate_not_applied" if policy_id != "M_KEEP_ALL" else "baseline_no_cleaning",
        "membership_before_rows": len(membership),
        "membership_after_rows_candidate": len(kept_ids),
        "membership_removed_rows_candidate": len(membership) - len(kept_ids),
        "raw_viewhistory_rows": len(views),
        "joined_temporal_rows_candidate": joined_count,
        "join_expansion_rows_vs_raw_viewhistory": joined_count - len(views),
        "join_expansion_ratio_vs_raw_viewhistory": round(joined_count / len(views), 6),
        "membership_rows_with_multiple_USER_NUM": multi_usernum_rows,
        "membership_rows_without_USER_NUM": no_usernum_rows,
        "recommendation": "compare only; do not apply silently",
    })

# Candidate rule summaries and decision matrix.
candidate_rule_summary = []
def add_rule(rule_id, domain, rule_status, decision_candidate, affected_rows, before_rows, after_rows_candidate, reason_code, recommendation, evidence_file):
    candidate_rule_summary.append({
        "rule_id": rule_id,
        "domain": domain,
        "rule_status": rule_status,
        "decision_candidate": decision_candidate,
        "before_rows": before_rows,
        "affected_rows": affected_rows,
        "after_rows_if_applied": after_rows_candidate,
        "reason_code": reason_code,
        "recommendation": recommendation,
        "evidence_file": evidence_file,
    })
add_rule("R_LABEL_01", "target_label", "candidate_not_applied", "ask_mentor", len(strict_conflict_ids), len(membership), len(membership)-len(strict_conflict_ids), "STRICT_TARGET_CONFLICT", "Do not choose labels for performance; exclude or flag only after mentor/business rule.", "02_v2_strict_conflict_group_audit.csv")
add_rule("R_DUP_01", "duplicates", "candidate_not_applied", "defer", len(exact_duplicate_extra_ids), len(membership), len(membership)-len(exact_duplicate_extra_ids), "EXACT_DUPLICATE_EXTRA_ROW", "Exact duplicate extras are plausible dedupe candidates after target conflict handling is fixed.", "02_v2_duplicate_group_audit.csv")
add_rule("R_DUP_02", "duplicates", "candidate_not_applied", "flag", len(core_ambiguous_ids), len(membership), len(membership), "CORE_EVENT_AMBIGUOUS", "Flag, sample, and review; do not drop silently.", "02_v2_duplicate_group_sample_rows.csv")
for row in duration_policy_comparison:
    add_rule(row["policy_id"], "duration", "candidate_not_applied", "defer", row["excluded_rows"], row["before_rows"], row["kept_rows"], "DURATION_POLICY_EXCLUSION_CANDIDATE", row["recommendation"], "02_v2_duration_policy_comparison.csv")
add_rule("R_MAP_01", "UserMapping", "candidate_not_applied", "flag", len(usermapping_one_to_many_audit), len(mapping), len(mapping), "USER_KEY_TO_MULTIPLE_USER_NUM", "Flag and aggregate logs back to membership_row_id in later stages.", "02_v2_usermapping_one_to_many_audit.csv")
add_rule("R_MAP_02", "UserMapping", "candidate_not_applied", "ask_mentor", len(usermapping_many_to_one_audit), len(mapping), len(mapping), "USER_NUM_TO_MULTIPLE_USER_KEY", "If present, ask mentor before USER_NUM grouping.", "02_v2_usermapping_many_to_one_audit.csv")
for row in value_anomaly_audit:
    add_rule(row["rule_id"], row["field"], "candidate_not_applied", row["decision_candidate"], row["affected_rows"], len(membership), len(membership), row["reason_code"], row["recommendation"], "02_v2_value_anomaly_audit.csv")
add_rule("APPLIED_NONE_01", "scope_guard", "applied", "keep", 0, len(membership), len(membership), "NO_ROWS_EXCLUDED_IN_STAGE02", "Stage 02 validates policy only; no preprocessing dataset is created.", "02_v2_final_checks.csv")

decision_matrix = []
for row in candidate_rule_summary:
    decision_matrix.append({
        "rule_id": row["rule_id"],
        "domain": row["domain"],
        "decision": row["decision_candidate"],
        "rule_status": row["rule_status"],
        "affected_rows": row["affected_rows"],
        "reason_code": row["reason_code"],
        "evidence_file": row["evidence_file"],
        "written_recommendation": row["recommendation"],
    })

candidate_policy_before_after_counts = []
for row in candidate_rule_summary:
    candidate_policy_before_after_counts.append({
        "rule_id": row["rule_id"],
        "rule_status": row["rule_status"],
        "before_rows": row["before_rows"],
        "affected_rows": row["affected_rows"],
        "after_rows_if_applied": row["after_rows_if_applied"],
        "changed_row_count_explanation": row["recommendation"],
        "reason_code": row["reason_code"],
    })

reason_code_catalog = []
for row in candidate_rule_summary:
    reason_code_catalog.append({
        "reason_code": row["reason_code"],
        "source_rule_id": row["rule_id"],
        "domain": row["domain"],
        "definition": row["recommendation"],
        "default_decision_candidate": row["decision_candidate"],
    })

# Candidate affected samples across rule types.
candidate_rule_affected_samples = []
for r in strict_conflict_sample_rows[:40]:
    candidate_rule_affected_samples.append({"rule_id": "R_LABEL_01", "reason_code": "STRICT_TARGET_CONFLICT", "membership_row_id": r["membership_row_id"], "USER_KEY": r["USER_KEY"], "sample_detail": f"target={r[TARGET_COL]}|reg={r['reg_date']}|end={r['end_date']}"})
for r in duplicate_group_sample_rows[:40]:
    candidate_rule_affected_samples.append({"rule_id": "R_DUP_01_OR_02", "reason_code": r["duplicate_type"], "membership_row_id": r["membership_row_id"], "USER_KEY": r["USER_KEY"], "sample_detail": f"target={r[TARGET_COL]}|reg={r['reg_date']}|end={r['end_date']}"})
for r in value_anomaly_samples[:80]:
    candidate_rule_affected_samples.append({"rule_id": r["rule_id"], "reason_code": r["reason_code"], "membership_row_id": r["membership_row_id"], "USER_KEY": r["USER_KEY"], "sample_detail": f"field={r['field']}|value={r['value']}|target={r['is_repurchase']}"})

# Save audit CSVs.
write_csv(TABLE_DIR / "02_v2_stage01_reread_inventory.csv", stage01_inventory, ["relative_path", "exists", "file_size_bytes", "last_write_time", "role"])
write_csv(TABLE_DIR / "02_v2_strict_conflict_group_audit.csv", strict_conflict_group_audit, ["conflict_group_id", "group_size", "target_values", "target_counts", "membership_row_ids", "recommended_action", "reason_code"])
write_csv(TABLE_DIR / "02_v2_strict_conflict_sample_rows.csv", strict_conflict_sample_rows, ["conflict_group_id", "sample_type", "membership_row_id"] + membership_cols)
write_csv(TABLE_DIR / "02_v2_duplicate_group_audit.csv", duplicate_group_audit, ["duplicate_type", "group_count", "row_count", "extra_rows", "recommendation", "reason_code"])
write_csv(TABLE_DIR / "02_v2_duplicate_group_sample_rows.csv", duplicate_group_sample_rows, ["duplicate_type", "group_id", "membership_row_id"] + membership_cols)
write_csv(TABLE_DIR / "02_v2_candidate_rule_summary.csv", candidate_rule_summary, ["rule_id", "domain", "rule_status", "decision_candidate", "before_rows", "affected_rows", "after_rows_if_applied", "reason_code", "recommendation", "evidence_file"])
write_csv(TABLE_DIR / "02_v2_candidate_rule_affected_samples.csv", candidate_rule_affected_samples, ["rule_id", "reason_code", "membership_row_id", "USER_KEY", "sample_detail"])
write_csv(TABLE_DIR / "02_v2_candidate_policy_before_after_counts.csv", candidate_policy_before_after_counts, ["rule_id", "rule_status", "before_rows", "affected_rows", "after_rows_if_applied", "changed_row_count_explanation", "reason_code"])
write_csv(TABLE_DIR / "02_v2_duration_policy_comparison.csv", duration_policy_comparison, ["policy_id", "policy_type", "before_rows", "kept_rows", "excluded_rows", "excluded_rate", "duration_distribution", "recommendation"])
write_csv(TABLE_DIR / "02_v2_duration_policy_samples.csv", duration_policy_samples, ["policy_id", "membership_row_id", "reason_code", "duration_days", "USER_KEY", "reg_date", "end_date", "is_repurchase"])
write_csv(TABLE_DIR / "02_v2_usermapping_one_to_many_audit.csv", usermapping_one_to_many_audit, ["USER_KEY", "mapping_row_count", "distinct_USER_NUM_count", "USER_NUM_values", "membership_event_count_for_USER_KEY", "would_duplicate_membership_rows_on_direct_join", "recommendation", "decision_candidate"])
write_csv(TABLE_DIR / "02_v2_usermapping_many_to_one_audit.csv", usermapping_many_to_one_audit, ["USER_NUM", "mapping_row_count", "distinct_USER_KEY_count", "USER_KEY_values", "recommendation", "decision_candidate"])
write_csv(TABLE_DIR / "02_v2_join_expansion_by_membership_policy.csv", join_expansion_by_policy, ["policy_id", "policy_type", "membership_before_rows", "membership_after_rows_candidate", "membership_removed_rows_candidate", "raw_viewhistory_rows", "joined_temporal_rows_candidate", "join_expansion_rows_vs_raw_viewhistory", "join_expansion_ratio_vs_raw_viewhistory", "membership_rows_with_multiple_USER_NUM", "membership_rows_without_USER_NUM", "recommendation"])
write_csv(TABLE_DIR / "02_v2_value_anomaly_audit.csv", value_anomaly_audit, ["rule_id", "field", "reason_code", "affected_rows", "affected_rate", "decision_candidate", "recommendation"])
write_csv(TABLE_DIR / "02_v2_value_anomaly_samples.csv", value_anomaly_samples, ["rule_id", "field", "reason_code", "membership_row_id", "USER_KEY", "value", "is_repurchase", "recommendation"])
write_csv(TABLE_DIR / "02_v2_decision_matrix.csv", decision_matrix, ["rule_id", "domain", "decision", "rule_status", "affected_rows", "reason_code", "evidence_file", "written_recommendation"])
write_csv(TABLE_DIR / "02_v2_reason_code_catalog.csv", reason_code_catalog, ["reason_code", "source_rule_id", "domain", "definition", "default_decision_candidate"])
write_csv(TABLE_DIR / "02_v2_row_exclusion_reason_candidates.csv", candidate_reason_rows, ["membership_row_id", "reason_code", "source_rule_id", "severity", "detail"])

# Markdown report and final checks.
raw_after = {name: {"size": path.stat().st_size, "mtime_ns": path.stat().st_mtime_ns} for name, path in RAW_FILES.items()}
required_outputs = [
    "02_v2_stage01_reread_inventory.csv", "02_v2_strict_conflict_group_audit.csv", "02_v2_strict_conflict_sample_rows.csv",
    "02_v2_duplicate_group_audit.csv", "02_v2_duplicate_group_sample_rows.csv", "02_v2_candidate_rule_summary.csv",
    "02_v2_candidate_rule_affected_samples.csv", "02_v2_candidate_policy_before_after_counts.csv", "02_v2_duration_policy_comparison.csv",
    "02_v2_duration_policy_samples.csv", "02_v2_usermapping_one_to_many_audit.csv", "02_v2_usermapping_many_to_one_audit.csv",
    "02_v2_join_expansion_by_membership_policy.csv", "02_v2_value_anomaly_audit.csv", "02_v2_value_anomaly_samples.csv",
    "02_v2_decision_matrix.csv", "02_v2_reason_code_catalog.csv", "02_v2_row_exclusion_reason_candidates.csv",
]
report_path = DATA_DIR / "02_v2_preprocessing_policy_validation_report.md"
summary_path = DATA_DIR / "02_v2_policy_validation_summary.json"
all_candidate_exclusions_have_reason_code = all(row.get("reason_code") for row in candidate_reason_rows)
allowed_decisions = {"apply", "flag", "keep", "defer", "ask_mentor"}
final_checks = [
    {"check": "stage01_outputs_reread", "status": "PASS" if all(p.exists() for p in STAGE01_REQUIRED) else "FAIL", "detail": f"checked={len(STAGE01_REQUIRED)}"},
    {"check": "raw_files_unchanged", "status": "PASS" if raw_before == raw_after else "FAIL", "detail": "raw file size and mtime unchanged"},
    {"check": "no_interim_dataset_created", "status": "PASS" if not (PROJECT_ROOT / "_data" / "02_interim" / "02_v2_preprocessing_policy_validation").exists() else "FAIL", "detail": "Stage 02 validation writes reports only"},
    {"check": "no_usage_features_created", "status": "PASS", "detail": "No usage feature table is written"},
    {"check": "no_model_trained", "status": "PASS", "detail": "No modeling library or training routine is used"},
    {"check": "strict_target_conflicts_validated", "status": "PASS" if len(strict_conflict_group_audit) > 0 else "FAIL", "detail": f"groups={len(strict_conflict_group_audit)}"},
    {"check": "duration_policies_compared", "status": "PASS" if len(duration_policy_comparison) >= 4 else "FAIL", "detail": f"policies={len(duration_policy_comparison)}"},
    {"check": "join_expansion_recomputed", "status": "PASS" if len(join_expansion_by_policy) >= 4 else "FAIL", "detail": f"policies={len(join_expansion_by_policy)}"},
    {"check": "decision_matrix_created", "status": "PASS" if len(decision_matrix) > 0 else "FAIL", "detail": f"rules={len(decision_matrix)}"},
    {"check": "decision_values_are_allowed", "status": "PASS" if all(row["decision"] in allowed_decisions for row in decision_matrix) else "FAIL", "detail": "allowed=apply|flag|keep|defer|ask_mentor"},
    {"check": "candidate_and_applied_rules_separated", "status": "PASS" if any(row["rule_status"] == "applied" for row in candidate_rule_summary) and any(row["rule_status"] == "candidate_not_applied" for row in candidate_rule_summary) else "FAIL", "detail": "applied scope guard is separate from candidate preprocessing rules"},
    {"check": "every_candidate_row_exclusion_has_reason_code", "status": "PASS" if all_candidate_exclusions_have_reason_code else "FAIL", "detail": f"candidate_reason_rows={len(candidate_reason_rows)}"},
    {"check": "all_required_outputs_created", "status": "PASS" if all((TABLE_DIR / name).exists() for name in required_outputs) else "FAIL", "detail": f"required_csvs={len(required_outputs)}"},
    {"check": "markdown_report_created", "status": "PASS" if report_path.exists() else "FAIL", "detail": rel(report_path)},
    {"check": "json_summary_created", "status": "PASS" if summary_path.exists() else "FAIL", "detail": rel(summary_path)},
]
write_csv(TABLE_DIR / "02_v2_final_checks.csv", final_checks, ["check", "status", "detail"])

report_lines = [
    "# 02_v2 Preprocessing Policy Validation Report", "",
    "## Scope", "- Re-read Stage 01 outputs and active v2 raw files.", "- Validate preprocessing candidates only.", "- No interim dataset, usage feature, modeling table, or trained model was created.", "",
    "## Applied Rules", "- `APPLIED_NONE_01`: no row exclusion or correction was applied in Stage 02. All proposed rules remain candidates.", "",
    "## Candidate Findings", f"- Strict target conflict groups: {len(strict_conflict_group_audit)} groups, {len(strict_conflict_ids)} rows.", f"- Exact duplicate groups: {len(exact_duplicate_groups)} groups.", f"- Core-event ambiguous groups: {len(core_event_ambiguous_groups)} groups.", f"- UserMapping one-to-many USER_KEY cases: {len(usermapping_one_to_many_audit)}.", f"- UserMapping many-to-one USER_NUM cases: {len(usermapping_many_to_one_audit)}.", "",
    "## Duration Policy Validation", "- Compared keep-all-parseable, positive-only, 31-or-32-only, and 28-to-35-day policies.", "- No duration policy was applied. Final policy is deferred until business definition of subscription duration and end_date inclusiveness is confirmed.", "",
    "## Decision Matrix", "- Decisions are limited to `apply`, `flag`, `keep`, `defer`, and `ask_mentor` candidates.", "- Current applied decision is only `keep` for the scope guard, meaning no preprocessing output is produced.", "",
    "## Row Exclusion Reason Codes", f"- Candidate row-exclusion reason rows: {len(candidate_reason_rows)}.", "- Every candidate row-exclusion row has a reason_code in `02_v2_row_exclusion_reason_candidates.csv`.", "",
    "## Output Files", 
]
for name in required_outputs + ["02_v2_final_checks.csv"]:
    report_lines.append(f"- {rel(TABLE_DIR / name)}")
report_lines.append(f"- {rel(summary_path)}")
report_lines.append(f"- {rel(report_path)}")
report_lines.extend(["", "## Final Checks"])
for row in final_checks:
    report_lines.append(f"- {row['check']}: {row['status']} ({row['detail']})")
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

write_json(summary_path, {
    "scope": "Stage 02 preprocessing policy validation only; no preprocessing dataset created.",
    "raw_row_counts": {name: len(info["rows"]) for name, info in loaded.items()},
    "strict_target_conflict_groups": len(strict_conflict_group_audit),
    "strict_target_conflict_rows": len(strict_conflict_ids),
    "exact_duplicate_groups": len(exact_duplicate_groups),
    "core_event_ambiguous_groups": len(core_event_ambiguous_groups),
    "duration_policy_count": len(duration_policy_comparison),
    "decision_rule_count": len(decision_matrix),
    "candidate_row_exclusion_reason_rows": len(candidate_reason_rows),
    "all_candidate_row_exclusions_have_reason_code": all_candidate_exclusions_have_reason_code,
    "final_checks": final_checks,
})

print("02_v2 preprocessing policy validation completed.")
print("Final checks:")
for row in final_checks:
    print(f"{row['check']}: {row['status']} - {row['detail']}")


02_v2 preprocessing policy validation completed.
Final checks:
stage01_outputs_reread: PASS - checked=11
raw_files_unchanged: PASS - raw file size and mtime unchanged
no_interim_dataset_created: PASS - Stage 02 validation writes reports only
no_usage_features_created: PASS - No usage feature table is written
no_model_trained: PASS - No modeling library or training routine is used
strict_target_conflicts_validated: PASS - groups=35
duration_policies_compared: PASS - policies=4
join_expansion_recomputed: PASS - policies=6
decision_matrix_created: PASS - rules=25
decision_values_are_allowed: PASS - allowed=apply|flag|keep|defer|ask_mentor
candidate_and_applied_rules_separated: PASS - applied scope guard is separate from candidate preprocessing rules
every_candidate_row_exclusion_has_reason_code: PASS - candidate_reason_rows=144
all_required_outputs_created: PASS - required_csvs=18
markdown_report_created: PASS - park.ingyeom/reports/data/02_v2_preprocessing_policy_validation/02_v2_preproc